In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 24.2 MB/s eta 0:00:00


In [ ]:
!uv pip install -r https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/requirements.txt --system

Using Python 3.12.13 environment at: /usr
Resolved 133 packages in 1.40s
Prepared 7 packages in 1.59s
Installed 7 packages in 343ms
 + async-lru==2.3.0
 + jedi==0.20.0
 + json5==0.15.0
 + jupyter-builder==1.2.2
 + jupyter-lsp==2.3.1
 + jupyterlab==4.6.3
 + jupyterlab-server==2.28.0


In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))



torch version: 2.11.0+cpu
tiktoken version: 0.13.0


In [ ]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 





*   The goal is to tokenize and embed this text for an LLM
Let's develop a simple tokenizer based on some simple sample text that we can then later apply to the text above
*   The following regular expression will split on whitespaces



In [ ]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)



['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']




*   Não queremos apenas separar espaços em brancos, mas também vírgulas e pontos. Então para isso mudamos a expressão regular



In [ ]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']




*   Como podemos ver essa expressão guarda também espaços vazios, vamos tirar-los



In [ ]:
# Tirar espaços em branco de cada item do vetor e então filtramos os espaços vazios
result = [item for item in result if item.strip()]
print(result)



['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']




*   Está ficando ótimo, mas agora iremos trabalhar com outros tipos de pontuações, como ponto de interrogação, pontos finais, entre outros.



In [ ]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']




*   Tudo indo bem, agora estamos prontos para aplicar a tokenização ao texto puro.



In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']




*   Após obtermos os tokens separados, iremos agora fazer a contagem deles.

In [ ]:
print(len(preprocessed))

4690




*   Vamos agora transformar tokens de texto em IDs de tokens
*   Agora com todos esses tokens teremos podemos montar um vocabulário que consiste em todos esses tokens unicos


In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [ ]:
vocab = {token:integer for integer,token in enumerate(all_words)}



*   Aqui a baixo, as 50 primeiras entradas do vocabulario


In [ ]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


*    Agora adicionaremos os tokens em uma classe.



In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

*   O Encoder será usado para transformar tokens de texto em tokens de ID
*   O Decoder fará o contrario.



*   Podemos usar a tokenização para codificar, texto para inteiros
*   Esses inteiros serão "embeeds", colocados em vetores, que serão entradas para nossa LLM.



In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]




*   Agora iremos converter esses inteiros para texto.


In [ ]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [ ]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

**2.4 Adding special context tokens**



*   É útil para nós adicionar alguns tokens especiais, para palavras desconhecidas e denotar o final de um texto
*   Alguns tokenizers usam tokens especiais para ajudar LLM com contexto adicional
*   Alguns tokens especiais são:
    +   [BOS] começo de uma sequência, começo de um texto;
    +   [EOS] final de um sequência, marca onde acaba o texto, para marcarmos quando tivermos dois textos de origem diferentes.
    +   [PAD] tokens de peenchimento, se treinarmos uma LLM com um tamanho de lote maior que 1, incluir texto com comprmentos diferentes. Com o token de padding, preenchemos os textos mais curtos até atingirem o comprimento do texto mais longo, de modo que todos os textos tenham o mesmo comprimento.
    +   [UNK] representa palavras que não estão incluidas no vocabulário.

*   GPT-2 apenas utilizam apenas o [EOS], as GPT's utilizam [EOS] para padding (preenchimento), já que usamos uma máscara quando estamos treinando com entradas em lote (batched inputs), não prestaríamos atenção aos tokens de padding de qualquer maneira. Portanto, não importa quais sejam esses tokens.
*   GPT-2 não utiliza tokens para palavras fora do vocabulário, em vez disso, ele utiliza um tokenizador baseado em Byte-Pair Enconding (BPE), divide as palavras em unidades de subpalavras (subword units)






In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'



*   Erro do sistema, aconteceu por conta da palavra "Hello" que não esá contida no vocabulário.
*   Para lidar com esses casos, adicionamos tokens especiais como "<|unk|>" ao vocabulário para representar as palavras desconhecidas.
*   Vamos também adicionar outro token chamado "<|endoftext|>", o qual, será usado no GPT-2 treinado para denotar o fim do texto e também para quando se coloca mais de um texto com fontes diferentes.




In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [ ]:
len(vocab.items())

1132

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)




*   Agora precisamos configurar um tokenizador em conformidade com os tokens adicionais, para que esse tokenizador entenda como usar os novos tokens.



In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Vamos simular!

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [ ]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [ ]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

**2.5 BytePair encoding**



*   GPT-2 usa o BytePair encoding (BPE) como um tokenizador.
*   Isso permite o modelo quebrar palavras que não estão pré definidas no vocabulário em pequenas unidades de subpalavras ou até caracteres, permitindo lidar com palavras de fora do vocabulário.
*   Por exemplo, se o vocabulário do GPT-2 não tiver a palavra "unfamiliarword", isso deve ser tokenizado como ["unfam", "iliar", "word"] ou alguma outra quebra de subpalavra, dependendo da mesclagem do treinamento BPE.
*   Tokenizador BPE original: https://github.com/openai/gpt-2/blob/master/src/encoder.py
*   Usaremos tokenizadores da biblioteca da OpenAI's open source tiktoken, a qual implementa algoritimos de núcleo em Rust para aprimorar a performance computacional.



In [ ]:
#pip install tiktoken

In [ ]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.13.0


In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [ ]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.




*   Tokenizadores BPE quebram palavras desconhecidas em subpalavras e caracteres individuais.



**2.6 Amostragem de dados com uma janela deslizante**


*   e train LLMs to generate one word at a time, so we want to prepare the training data accordingly where the next word in a sequence represents the target to predict:
*   Treinamos uma LLM para gerar uma palavra por vez, então queremos preparar o treinamento de dados, conforme onde estará a próxima palavra na sequência representada do alvo da premedição.

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145




*   para cada pedaço de texto, nós queremos os inputs e alvos
*   Como queremos que nosso modelo premedite a próxima palavra, os alvos são inputs deslocados para a posição a direita.



In [ ]:
enc_sample = enc_text[50:]

In [ ]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


*   Um por um, deve parecer com:

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


*    Implantaremos um data loader que interage em dataset de entrada e retorna os alvos deslocados por um.

In [ ]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu




*   Usamos abordagem de janelas deslizantes, mudando a posição por +1.
*   Criamos um dataset e um data loader que extrai pedaços do dataset de texto de entrada



In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]



In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader





*   Vamos testar o dataloader com o lote de tamanho 1 para uma LLM com contexto de tamanho 4.



In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [ ]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


*    Um Exemplo usando um passo igual ao tamanho do contexto (4).



*   Podemos criar lotes de saídas também;
*   Note que aumentamos o passo aqui para que não tenhamos sobreposições entre os lotes, uma vez que mais sobreposição pode levar ao aumento do excesso.

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


**2.7 Creating token embeddings**

*   Os dados já estão quase prontos para um LLM
*   Mas, por último, vamos incorporar os tokens em uma representação vetorial contínua usando uma camada de incorporação
*   Normalmente, essas camadas de incorporação fazem parte do próprio LLM e são atualizadas (treinadas) durante o treinamento do modelo
*   Suponha que temos os seguintes quatro exemplos de entrada com ids de entrada 2, 3, 5 e 1 (após a tokenização):

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

*   Por uma questão de simplicidade, suponha que temos um pequeno vocabulário de apenas 6 palavras e queremos criar incorporações de tamanho 3:

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)



*   Isso resultaria em uma matriz de peso 6x3:



In [ ]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


*   Para aqueles que estão familiarizados com a codificação one-hot, a abordagem de
camada de incorporação acima é essencialmente apenas uma maneira mais eficiente de implementar codificação one-hot seguida de multiplicação de matriz em uma camada totalmente conectada
*   Como a camada de incorporação é apenas uma implementação mais eficiente, equivalente à abordagem de codificação e multiplicação de matrizes, ela pode ser vista como uma camada de rede neural que pode ser otimizada via retropropagação
*   Para converter um token com id 3 em um vetor de 3 dimensões, fazemos o seguinte:

In [ ]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


*   Observe que a acima é a 4a linha na matriz de peso embedding_layer
*   Para incorporar todos os quatro valores de input_ids acima, fazemos

In [ ]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)




*   Uma camada de incorporação é essencialmente uma operação de look-up:


**2.8 Encoding word positions**



*   Embutindo ideias de conversão de camada em representações vetoriais idênticas, independentemente de onde eles estão localizados na sequência de entrada
*   Incorporações posicionais são combinadas com o vetor de incorporação de token para formar as incorporações de entrada para um modelo de linguagem grande
*   O codificador BytePair tem um tamanho de vocabulário de 50.257
*   Suponha que queremos codificar os tokens de entrada em uma representação vetorial de 256 dimensões



In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

*    Se amostrarmos dados do dataloader, incorporamos os tokens em cada lote em um vetor de 256 dimensões
*    Se tivermos um tamanho de lote de 8 com 4 fichas cada, isso resulta em um tensor de 8 x 4 x 256:

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [ ]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(token_embeddings)

torch.Size([8, 4, 256])




*   O GPT-2 usa incorporações de posição absoluta, então apenas criamos outra camada de incorporação

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
# print(pos_embedding_layer.weight)

In [ ]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(pos_embeddings)

torch.Size([4, 256])




*   Para criar as incorporações de entrada usadas em um LLM, simplesmente adicionamos o token e as incorporações posicionais



In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(input_embeddings)

torch.Size([8, 4, 256])


*   Na fase inicial do fluxo de trabalho de processamento de entrada, o texto de entrada é segmentado em tokens separados
*   Após essa segmentação, esses tokens são transformados em IDs de token com base em um vocabulário predefinido

# 2.9 Experimentação (Sprint 2 — Seção 4)

*   A partir do pipeline construído acima (tokenização → vocabulário → Token IDs → sequências → embeddings → positional embeddings → DataLoader), vamos variar os principais parâmetros e observar o impacto sobre os dados e suas representações.
*   Os experimentos usam o mesmo texto `the-verdict.txt` já carregado em `raw_text`, o vocabulário `vocab` construído anteriormente e o tokenizador BPE (`tokenizer`, GPT-2 via `tiktoken`).

**Experimento 1 — Quantidade de tokens produzidos para textos diferentes (BPE)**

*   Comparamos a contagem de tokens BPE gerada para uma frase curta, um parágrafo médio e o texto completo `the-verdict.txt`.

In [ ]:
textos = {
    "Frase curta": "Hello, do you like tea?",
    "Paragrafo medio": raw_text[:500],
    "the-verdict.txt completo": raw_text,
}
for nome, txt in textos.items():
    ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
    print(f"{nome:30s} | caracteres: {len(txt):6d} | tokens BPE: {len(ids):6d} | razao chars/token: {len(txt)/max(len(ids),1):.2f}")

Frase curta                    | caracteres:     23 | tokens BPE:      7 | razao chars/token: 3.29
Paragrafo medio                | caracteres:    500 | tokens BPE:    123 | razao chars/token: 4.07
the-verdict.txt completo       | caracteres:  20479 | tokens BPE:   5145 | razao chars/token: 3.98


**Experimento 2 — Tokenizador por palavras (SimpleTokenizerV2) vs BPE, mesma sequência**

*   O mesmo trecho é codificado pelos dois tokenizadores para comparar a granularidade e o tamanho de vocabulário de cada abordagem.

In [ ]:
word_tokenizer = SimpleTokenizerV2(vocab)

sample = "In the sunlit terraces of the palace, Mrs. Gisburn said with pardonable pride."
word_ids = word_tokenizer.encode(sample)
bpe_ids = tokenizer.encode(sample)

print(f"Texto: {sample!r}")
print(f"Tokens (palavras, vocab={len(vocab)}): {len(word_ids)} -> {word_ids}")
print(f"Tokens (BPE, vocab=50257):          {len(bpe_ids)} -> {bpe_ids}")

Texto: 'In the sunlit terraces of the palace, Mrs. Gisburn said with pardonable pride.'
Tokens (palavras, vocab=1132): 16 -> [55, 988, 956, 984, 722, 988, 1131, 5, 67, 7, 38, 851, 1108, 754, 793, 7]
Tokens (BPE, vocab=50257):          21 -> [818, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 11, 9074, 13, 402, 271, 10899, 531, 351, 27322, 540, 11293, 13]


**Experimento 3 — Relação entre Context Size e quantidade de amostras**

*   Fixamos `stride = context_size` (sem sobreposição) e variamos o tamanho do contexto, observando quantas amostras de treinamento o mesmo texto produz.

In [ ]:
for ctx in [2, 4, 8, 16, 32, 64, 128]:
    dl = create_dataloader_v1(raw_text, batch_size=1, max_length=ctx, stride=ctx, shuffle=False)
    n_amostras = len(dl.dataset)
    print(f"context_size={ctx:4d} | stride={ctx:4d} | amostras produzidas: {n_amostras:5d}")

context_size=   2 | stride=   2 | amostras produzidas:  2572
context_size=   4 | stride=   4 | amostras produzidas:  1286
context_size=   8 | stride=   8 | amostras produzidas:   643
context_size=  16 | stride=  16 | amostras produzidas:   321
context_size=  32 | stride=  32 | amostras produzidas:   160
context_size=  64 | stride=  64 | amostras produzidas:    80
context_size= 128 | stride= 128 | amostras produzidas:    40


**Experimento 4 — Impacto do Stride (sobreposição) com Context Size fixo**

*   Fixamos `context_size = 32` e variamos o `stride`, observando como a sobreposição entre janelas afeta a quantidade de amostras.

In [ ]:
ctx = 32
for stride in [8, 16, 32, 64]:
    dl = create_dataloader_v1(raw_text, batch_size=1, max_length=ctx, stride=stride, shuffle=False)
    n_amostras = len(dl.dataset)
    overlap_pct = max(0, (ctx - stride) / ctx * 100)
    print(f"context_size={ctx} | stride={stride:4d} | amostras: {n_amostras:5d} | overlap: {overlap_pct:.0f}%")

context_size=32 | stride=   8 | amostras:   640 | overlap: 75%
context_size=32 | stride=  16 | amostras:   320 | overlap: 50%
context_size=32 | stride=  32 | amostras:   160 | overlap: 0%
context_size=32 | stride=  64 | amostras:    80 | overlap: 0%


**Experimento 5 — Diferentes Batch Sizes (shape dos lotes)**

*   Mantemos `context_size = 4` e variamos o `batch_size`, observando o número de lotes e o shape dos tensores de entrada/alvo.

In [ ]:
for bs in [1, 4, 8, 16, 32]:
    dl = create_dataloader_v1(raw_text, batch_size=bs, max_length=4, stride=4, shuffle=False, drop_last=True)
    n_batches = len(dl)
    batch_inputs, batch_targets = next(iter(dl))
    print(f"batch_size={bs:3d} | n_batches={n_batches:4d} | shape inputs: {tuple(batch_inputs.shape)} | shape targets: {tuple(batch_targets.shape)}")

batch_size=  1 | n_batches=1286 | shape inputs: (1, 4) | shape targets: (1, 4)
batch_size=  4 | n_batches= 321 | shape inputs: (4, 4) | shape targets: (4, 4)
batch_size=  8 | n_batches= 160 | shape inputs: (8, 4) | shape targets: (8, 4)
batch_size= 16 | n_batches=  80 | shape inputs: (16, 4) | shape targets: (16, 4)
batch_size= 32 | n_batches=  40 | shape inputs: (32, 4) | shape targets: (32, 4)


**Experimento 6 — Diferentes dimensões de Embedding (parâmetros e shape)**

*   Reutilizamos o mesmo lote (`batch_size=8`, `context_size=4`) e variamos `output_dim` para observar o impacto na quantidade de parâmetros das camadas de embedding e no shape do `input_embeddings` final.

In [ ]:
vocab_size_bpe = tokenizer.n_vocab
exp_max_length = 4
exp_dl = create_dataloader_v1(raw_text, batch_size=8, max_length=exp_max_length, stride=exp_max_length, shuffle=False)
exp_inputs, exp_targets = next(iter(exp_dl))

for out_dim in [8, 32, 64, 128, 256, 768]:
    torch.manual_seed(123)
    exp_token_emb = torch.nn.Embedding(vocab_size_bpe, out_dim)
    exp_pos_emb = torch.nn.Embedding(exp_max_length, out_dim)
    n_params_token = sum(p.numel() for p in exp_token_emb.parameters())
    n_params_pos = sum(p.numel() for p in exp_pos_emb.parameters())
    exp_input_emb = exp_token_emb(exp_inputs) + exp_pos_emb(torch.arange(exp_max_length))
    print(f"output_dim={out_dim:4d} | shape input_embeddings: {tuple(exp_input_emb.shape)} | params token_emb: {n_params_token:>10,} | params pos_emb: {n_params_pos:>8,}")

output_dim=   8 | shape input_embeddings: (8, 4, 8) | params token_emb:    402,056 | params pos_emb:       32
output_dim=  32 | shape input_embeddings: (8, 4, 32) | params token_emb:  1,608,224 | params pos_emb:      128
output_dim=  64 | shape input_embeddings: (8, 4, 64) | params token_emb:  3,216,448 | params pos_emb:      256
output_dim= 128 | shape input_embeddings: (8, 4, 128) | params token_emb:  6,432,896 | params pos_emb:      512
output_dim= 256 | shape input_embeddings: (8, 4, 256) | params token_emb: 12,865,792 | params pos_emb:    1,024
output_dim= 768 | shape input_embeddings: (8, 4, 768) | params token_emb: 38,597,376 | params pos_emb:    3,072


**Experimento 7 — Custo computacional: BPE vs tokenização por palavras**

*   Medimos o tempo médio de tokenização do texto completo com os dois métodos, para observar a diferença de custo entre eles.

In [ ]:
import time

n_rep = 20

t0 = time.perf_counter()
for _ in range(n_rep):
    tokenizer.encode(raw_text)
t_bpe = (time.perf_counter() - t0) / n_rep

t0 = time.perf_counter()
for _ in range(n_rep):
    word_tokenizer.encode(raw_text.replace("\n", " "))
t_word = (time.perf_counter() - t0) / n_rep

print(f"Tempo medio BPE (tiktoken)      : {t_bpe*1000:.3f} ms  ({n_rep} execucoes)")
print(f"Tempo medio palavras (regex V2) : {t_word*1000:.3f} ms  ({n_rep} execucoes)")

Tempo medio BPE (tiktoken)      : 3.769 ms  (20 execucoes)
Tempo medio palavras (regex V2) : 6.387 ms  (20 execucoes)


# 2.10 Análise dos resultados

*   A análise técnica completa, respondendo às perguntas da Seção 5 do roteiro da Sprint 2 com base nos experimentos acima, está em [`docs/sprint2_analise.md`](../docs/sprint2_analise.md).